# Getting Started with Alpaca Paper Trading
### SIGecom Algo-Trading Project, Fall 2026

This notebook is your first hands-on look at **Alpaca**, the brokerage API we use for the project. Work through it on your own. By the end you will have:

1. Set up your own **Alpaca paper trading account** (fake money, real market data).
2. Generated API keys and stored them safely.
3. Connected to Alpaca from Python and checked your account.
4. Pulled historical and live market data.
5. Placed, checked, and cancelled orders, and managed positions.

> **Paper trading only.** Everything here uses simulated money. You don't need to deposit anything, and you should **not** enter bank details or fund a live account for this project.

---
## Part 1: What is Alpaca?

[Alpaca](https://alpaca.markets) is a commission-free brokerage built for programmatic trading. Instead of clicking buttons in an app, you send orders through a **REST API**. Alpaca gives every account two environments:

| | **Paper** | **Live** |
|---|---|---|
| Money | Simulated (starts at $100,000 by default) | Real |
| Market data | Real | Real |
| Order fills | Simulated against real quotes | Real exchange fills |
| API base URL | `https://paper-api.alpaca.markets` | `https://api.alpaca.markets` |
| API keys | Paper keys | Live keys (different!) |

Paper and live keys are **separate**. Paper keys can't touch real money, so they're the only keys you need for this project.

---
## Part 2: Create your paper trading account

Alpaca updates its dashboard from time to time, so button labels may be slightly different from what's written here. The steps stay the same.

### Step 1: Sign up
1. Go to **https://app.alpaca.markets/signup**.
2. Sign up with your email and a strong password. (Your school email is fine.)
3. Open the verification email from Alpaca and click the link to confirm your address.

A paper-only account needs nothing more than a verified email. **You do not need to finish the identity verification / KYC steps** (SSN, address, employment, and so on). Those are only for opening a *live* account. If the dashboard asks you to "complete your application," you can skip it.

### Step 2: Switch to your paper account
1. Log in at **https://app.alpaca.markets**.
2. Find the **account switcher** in the top-left corner of the dashboard. It shows either your live account or a paper account (paper account numbers start with **`PA`**).
3. Select your **Paper Trading** account. The dashboard should now show a balance of about **$100,000**.

> Always check that you're on the **paper** account before generating keys. Everything below assumes you are.

### Step 3: Generate API keys
1. On the paper account's home page, find the **API Keys** panel (usually on the right side).
2. Click **Generate New Keys** (or **Regenerate** if keys already exist).
3. You'll see two values:
   - **API Key ID**, for example `PKXXXXXXXXXXXXXXXXXX` (paper keys usually start with `PK`)
   - **Secret Key**, a longer random string
4. **Copy both right away.** Alpaca shows the Secret Key **only once**. If you lose it, regenerate the keys (which invalidates the old pair).

### Step 4 (optional): Reset or resize your paper account
From the paper account menu you can **reset** the account, which wipes positions and history and lets you choose a new starting balance. This helps if you want to start over clean or test with a realistic amount (say $10,000 instead of $100,000).

> Paper accounts follow the **Pattern Day Trader (PDT)** rule just like real ones: if equity drops below **$25,000**, you're limited to 3 day trades in any rolling 5 business days. Keep this in mind if you shrink your balance.

---
## Part 3: Keep your keys safe

API keys are passwords. Anyone who has them can trade in your account. Even though this is paper money, practice good habits now:

- **Never** paste keys directly into a notebook cell or `.py` file.
- **Never** commit keys to Git. Bots scrape GitHub for leaked keys within minutes.
- Store them in a **`.env`** file that Git ignores.

### Create a `.env` file
In the **same folder as this notebook**, create a file named exactly `.env` (no filename before the dot) with these contents, replacing the placeholders with your keys:

```
ALPACA_API_KEY=PKXXXXXXXXXXXXXXXXXX
ALPACA_SECRET_KEY=your_secret_key_here
```

No quotes, and no spaces around the `=`. This repo's `.gitignore` already ignores `.env`, so it won't be committed. You can copy `.env.example` to `.env` as a starting point.

---
## Part 4: Install the Python packages

We use **[`alpaca-py`](https://alpaca.markets/sdks/python/)**, Alpaca's official Python SDK.

> ⚠️ You may see older tutorials that use `alpaca-trade-api` or `import alpaca_trade_api`. That library is **deprecated**. Use `alpaca-py` (imported as `alpaca`).

Run the cell below once. If you use a virtual environment or conda env, make sure Jupyter is running inside it.

In [ ]:
%pip install --quiet alpaca-py python-dotenv pandas matplotlib

---
## Part 5: Load your keys and connect

The cell below reads your keys from `.env`. If it can't find them, it asks you to type them in instead (they won't be saved anywhere).

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()  # reads .env in the current working directory (or a parent)

API_KEY = os.getenv("ALPACA_API_KEY") or getpass("Alpaca API Key ID: ")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY") or getpass("Alpaca Secret Key: ")

assert API_KEY and SECRET_KEY, "Missing Alpaca keys. See Part 3."
print("Keys loaded. API key starts with:", API_KEY[:4] + "...")

`alpaca-py` splits functionality across several clients. The two you'll use most are:

- **`TradingClient`**: account info, orders, positions, market clock
- **`StockHistoricalDataClient`**: price bars, quotes, trades

Passing `paper=True` sends every request to the paper endpoint. **Always include it** in this project.

In [ ]:
from alpaca.trading.client import TradingClient
from alpaca.data.historical import StockHistoricalDataClient

trading_client = TradingClient(API_KEY, SECRET_KEY, paper=True)
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

### Check your account
If this cell runs without errors, your keys work. If you get a `401 Unauthorized` / `forbidden` error, the usual causes are:
- live keys instead of paper keys,
- a typo or extra whitespace in `.env`,
- keys that were regenerated after you copied them.

In [ ]:
account = trading_client.get_account()

print(f"Account number : {account.account_number}")
print(f"Status         : {account.status}")
print(f"Cash           : ${float(account.cash):,.2f}")
print(f"Buying power   : ${float(account.buying_power):,.2f}")
print(f"Equity         : ${float(account.equity):,.2f}")
print(f"PDT flagged    : {account.pattern_day_trader}")

if not str(account.account_number).startswith("PA"):
    print("\n⚠️  This doesn't look like a paper account number. Double-check your keys!")

Note that numeric fields such as `cash` and `equity` come back as **strings**, so wrap them in `float()` before doing math.

---
## Part 6: The market clock

US equity markets are open **9:30 AM to 4:00 PM Eastern**, Monday to Friday, excluding holidays. Orders you submit outside those hours are usually **queued** until the next open instead of filling immediately. That's the most common source of "why didn't my order fill?" questions.

In [ ]:
clock = trading_client.get_clock()

print(f"Market open right now? {clock.is_open}")
print(f"Current time           : {clock.timestamp}")
print(f"Next open              : {clock.next_open}")
print(f"Next close             : {clock.next_close}")

### Look up an asset
Before trading a symbol, you can check whether Alpaca supports it and whether it can be bought in fractional shares.

In [ ]:
asset = trading_client.get_asset("AAPL")

print(f"{asset.symbol}: {asset.name}")
print(f"  Exchange     : {asset.exchange}")
print(f"  Tradable     : {asset.tradable}")
print(f"  Fractionable : {asset.fractionable}")
print(f"  Shortable    : {asset.shortable}")

---
## Part 7: Market data

### A note on data feeds
The free Alpaca plan gives you:
- **IEX** feed: real-time data from the IEX exchange only (a small slice of total volume, but free and live).
- **SIP** feed: consolidated data from all exchanges, but on the free plan you can't request the **most recent 15 minutes**.

To avoid permission errors we request `feed=DataFeed.IEX` below. For historical research (backtests) you can switch to `DataFeed.SIP` as long as your end time is more than 15 minutes in the past.

### Historical bars
A **bar** (or candle) summarizes trading over one time interval: open, high, low, close, volume (OHLCV).

In [ ]:
from datetime import datetime, timedelta, timezone

from alpaca.data.enums import DataFeed
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

request = StockBarsRequest(
    symbol_or_symbols=["AAPL", "SPY"],
    timeframe=TimeFrame.Day,
    start=datetime.now(timezone.utc) - timedelta(days=60),
    feed=DataFeed.IEX,
)

bars = data_client.get_stock_bars(request)
df = bars.df  # pandas DataFrame indexed by (symbol, timestamp)
df.tail(10)

The DataFrame has a two-level index, `(symbol, timestamp)`. To work with a single symbol:

In [ ]:
spy = df.loc["SPY"].copy()
spy["return"] = spy["close"].pct_change()

print(f"SPY bars: {len(spy)}")
print(f"Average daily return : {spy['return'].mean():.4%}")
print(f"Daily volatility     : {spy['return'].std():.4%}")

spy["close"].plot(title="SPY daily close (IEX)", figsize=(10, 4));

Other timeframes work the same way, for example 15-minute bars:

```python
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
TimeFrame(15, TimeFrameUnit.Minute)
```

### Latest quote
A **quote** is the current best **bid** (highest price someone will pay) and **ask** (lowest price someone will sell at). The gap between them is the **spread**.

In [ ]:
from alpaca.data.requests import StockLatestQuoteRequest

quote = data_client.get_stock_latest_quote(
    StockLatestQuoteRequest(symbol_or_symbols="AAPL", feed=DataFeed.IEX)
)["AAPL"]

print(f"AAPL bid: ${quote.bid_price:.2f} x {quote.bid_size}")
print(f"AAPL ask: ${quote.ask_price:.2f} x {quote.ask_size}")
print(f"Spread  : ${quote.ask_price - quote.bid_price:.2f}")
print(f"As of   : {quote.timestamp}")

---
## Part 8: Placing orders

This is the main event. All orders go through `trading_client.submit_order(...)`, which takes an order *request* object.

### Order types you'll use most

| Type | What it does | When to use |
|---|---|---|
| **Market** | Buy or sell immediately at the best available price | You care about getting filled, not the exact price |
| **Limit** | Buy at or below (or sell at or above) a price you set | You care about price, and are OK not getting filled |

### Time in force
- `DAY`: cancelled automatically at market close if not filled.
- `GTC` (good 'til cancelled): stays open until filled or you cancel it.

> Fractional and notional (dollar-amount) orders must use `DAY`.

### 8.1 Market order: buy 1 share of SPY
If the market is closed, this order will show status `accepted` and fill at the next open.

In [ ]:
from alpaca.trading.enums import OrderSide, TimeInForce
from alpaca.trading.requests import MarketOrderRequest

market_order = trading_client.submit_order(
    order_data=MarketOrderRequest(
        symbol="SPY",
        qty=1,
        side=OrderSide.BUY,
        time_in_force=TimeInForce.DAY,
    )
)

print(f"Order ID : {market_order.id}")
print(f"Status   : {market_order.status}")

Orders are **asynchronous**. `submit_order` returns as soon as Alpaca *accepts* the order, not when it fills. To see what happened, look the order up again:

In [ ]:
import time

time.sleep(2)  # give it a moment to fill if the market is open
order = trading_client.get_order_by_id(market_order.id)

print(f"Status         : {order.status}")
print(f"Filled qty     : {order.filled_qty}")
print(f"Filled avg px  : {order.filled_avg_price}")

### 8.2 Notional (dollar-amount) order: buy $500 of AAPL
Instead of a share count, you can specify a dollar amount and Alpaca buys fractional shares. This only works for assets with `fractionable=True`.

In [ ]:
notional_order = trading_client.submit_order(
    order_data=MarketOrderRequest(
        symbol="AAPL",
        notional=500,
        side=OrderSide.BUY,
        time_in_force=TimeInForce.DAY,
    )
)

print(f"Notional order {notional_order.id}: {notional_order.status}")

### 8.3 Limit order
Here we place a buy limit order **well below** the current price, so it almost certainly won't fill. That lets us practice checking and cancelling open orders.

In [ ]:
from alpaca.trading.requests import LimitOrderRequest

limit_price = round(quote.bid_price * 0.90, 2)  # 10% below the current bid

limit_order = trading_client.submit_order(
    order_data=LimitOrderRequest(
        symbol="AAPL",
        qty=1,
        side=OrderSide.BUY,
        time_in_force=TimeInForce.GTC,
        limit_price=limit_price,
    )
)

print(f"Limit buy AAPL @ ${limit_price}: {limit_order.status}  (id={limit_order.id})")

### 8.4 List open orders

In [ ]:
from alpaca.trading.enums import QueryOrderStatus
from alpaca.trading.requests import GetOrdersRequest

open_orders = trading_client.get_orders(
    filter=GetOrdersRequest(status=QueryOrderStatus.OPEN)
)

for o in open_orders:
    print(f"{o.symbol:6} {o.side.value:4} qty={o.qty} notional={o.notional} "
          f"type={o.order_type.value:6} limit={o.limit_price} status={o.status.value}")

### 8.5 Cancel an order

In [ ]:
trading_client.cancel_order_by_id(limit_order.id)

time.sleep(1)
print("Limit order status:", trading_client.get_order_by_id(limit_order.id).status)

To cancel **every** open order at once, use `trading_client.cancel_orders()`.

---
## Part 9: Positions

A **position** is what you currently hold. Once your SPY and AAPL orders fill (immediately if the market is open, otherwise at the next open), they show up here.

In [ ]:
positions = trading_client.get_all_positions()

if not positions:
    print("No open positions yet. If the market is closed, your orders fill at the next open.")

for p in positions:
    print(f"{p.symbol:6} qty={float(p.qty):>10.4f}  avg entry=${float(p.avg_entry_price):>9.2f}  "
          f"market value=${float(p.market_value):>10.2f}  unrealized P/L=${float(p.unrealized_pl):>8.2f}")

### Close positions
`close_position` sells (or buys back, for shorts) your entire position in one symbol with a market order. `close_all_positions` flattens everything.

Uncomment one of these when you want to close out:

In [ ]:
# trading_client.close_position("SPY")
# trading_client.close_all_positions(cancel_orders=True)

---
## Part 10: Things to know about paper trading

Paper trading is great for learning, but it's **more forgiving than reality**. Keep these in mind before trusting paper results:

- **No market impact.** Paper fills don't move the price. A real order for 10,000 shares would.
- **Optimistic fills.** Paper orders fill against the current quote; in real markets you may get partial fills or worse prices (slippage), especially in thin stocks.
- **No dividends or corporate-action cash** is simulated on paper.
- **Rate limits.** The API allows about **200 requests per minute** per account. Don't poll in a tight loop; add `time.sleep(...)`.
- **Same rules as live.** Market hours, PDT, short-selling availability, and buying power all still apply.

### Useful links
- Alpaca dashboard: https://app.alpaca.markets
- `alpaca-py` docs: https://alpaca.markets/sdks/python/
- Trading API reference: https://docs.alpaca.markets/reference
- Paper trading docs: https://docs.alpaca.markets/docs/paper-trading

---
## Part 11: Try it yourself

Do these on your own to make sure the ideas stuck. Add new cells below.

1. **Portfolio snapshot.** Write a function `portfolio_summary()` that prints your equity, cash, and a table of every position with its percentage of total equity.
2. **Your own data.** Pull **1-hour bars** for three stocks you're interested in over the last 30 days. Which had the highest volatility?
3. **Order babysitter.** Place a limit buy for 1 share of a stock 2% below its current ask. Write a loop that checks the order every 10 seconds (for at most 2 minutes) and cancels it if it hasn't filled.
4. **A tiny strategy.** Using daily bars, compute the 10-day and 30-day moving averages of SPY. If the 10-day is above the 30-day, buy 1 share (if you don't already hold it); otherwise close any SPY position. Run it once by hand. *(Later in the project we'll talk about backtesting this properly before running anything live.)*
5. **Clean up.** Cancel all open orders and close all positions so your account is fresh.